# Relative abundance of _C. acnes_ MAGs (n=22) across shotgun samples

Date created: 10/25/2024

Notebook author: Yang Chen

Data analysis by: Yang Chen

In [284]:
# Import Python packages
import pandas as pd
import biom
import matplotlib.pyplot as plt
import seaborn as sns
import os
from itertools import cycle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

In [285]:
# Read in metadata
md = pd.read_csv('../Metadata/metadata_final_22102024.tsv', sep='\t')
md['zone'] = md['zone'].replace({'Lesional': 'L', 'Non-lesional': 'NL'})
md['label'] = md['subject_ID'].astype(str) + '_' + 'sev-'+ md['local_lesion_severity'].astype(str) + '_' + md['zone']
md = md.sort_values(by=['subject_ID', 'local_lesion_severity'])
md.index = md['SampleID']
md.head()

,SampleID,c_zone,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face,zone,sample_type,planned_study_day_of_visit,visual_assessment_in_vivo_number_of_inflammatory_lesions_face,day,subject_randomization_number,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_cheek_right,...,cohort,subject_randomization_id,class,subject_ID,subject_ID_CC,zone_CC,group,severity_level,severity_group,label
SampleID,,,,,,,,,,,,,,,,,,,,,
LAMI.RD001.D14.C1,LAMI.RD001.D14.C1,C1,not applicable,NL,skin,Day 14,not applicable,14,1,not applicable,...,control,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy,absent,absent Healthy,PP_1_sev-0_NL
LAMI.RD001.D28.C1,LAMI.RD001.D28.C1,C1,not applicable,NL,skin,Day 28,not applicable,28,1,not applicable,...,control,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy,absent,absent Healthy,PP_1_sev-0_NL
LAMI.RD001.D0.C2,LAMI.RD001.D0.C2,C2,not applicable,NL,skin,Day 0,not applicable,0,1,not applicable,...,control,RD001,healthy,PP_1,PP_1C2,Non-lesional_C2,Healthy,absent,absent Healthy,PP_1_sev-0_NL
LAMI.RD001.D28.C2,LAMI.RD001.D28.C2,C2,not applicable,NL,skin,Day 28,not applicable,28,1,not applicable,...,control,RD001,healthy,PP_1,PP_1C2,Non-lesional_C2,Healthy,absent,absent Healthy,PP_1_sev-0_NL
LAMI.RD001.D0.C1,LAMI.RD001.D0.C1,C1,not applicable,NL,skin,Day 0,not applicable,0,1,not applicable,...,control,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy,absent,absent Healthy,PP_1_sev-0_NL


In [286]:
md['label']

SampleID
LAMI.RD001.D14.C1    PP_1_sev-0_NL
LAMI.RD001.D28.C1    PP_1_sev-0_NL
LAMI.RD001.D0.C2     PP_1_sev-0_NL
LAMI.RD001.D28.C2    PP_1_sev-0_NL
LAMI.RD001.D0.C1     PP_1_sev-0_NL
                         ...      
LAMI.RD007.D0.C2     PP_7_sev-0_NL
LAMI.RD007.D14.C2    PP_7_sev-0_NL
LAMI.RD007.D0.C1     PP_7_sev-0_NL
LAMI.RD007.D28.C1    PP_7_sev-0_NL
LAMI.RD007.D28.C2    PP_7_sev-0_NL
Name: label, Length: 266, dtype: object

In [287]:
md['local_lesion_severity']

SampleID
LAMI.RD001.D14.C1    0
LAMI.RD001.D28.C1    0
LAMI.RD001.D0.C2     0
LAMI.RD001.D28.C2    0
LAMI.RD001.D0.C1     0
                    ..
LAMI.RD007.D0.C2     0
LAMI.RD007.D14.C2    0
LAMI.RD007.D0.C1     0
LAMI.RD007.D28.C1    0
LAMI.RD007.D28.C2    0
Name: local_lesion_severity, Length: 266, dtype: int64

In [288]:
# Read in input file
df = pd.read_csv('../Data/metaG/Tables/X_mags.with_taxonomy.with_slcs_c.acnes.csv')

# Assuming 'name_group' is the column containing the full names
# Extract the part that starts with "Cutibacterium acnes Type" and assign it to a new column 'name_group'
df['name.group'] = df['name'].str.extract(r'(Cutibacterium acnes Type \S+)')

# Subset to only C. acnes rows
df = df[df['name'].str.startswith(' s__Cutibacterium acnes', na=False)]

# Set 'name' as index and drop first 4 columns
df = df.set_index('name').drop(['SLC', 'sample_mag', 'organism_type', 'name.group', 'taxonomy'], axis=1)

df.index.name = None
df

,LAMI_RD308_D9_C3_S18_L005,LAMI_RD306_D28_C3_S12_L005,LAMI_RD310_D16_C2_S22_L005,LAMI_RD304_D14_C3_S4_L005,LAMI_RD304_D11_C1_S3_L005,LAMI_RD306_D14_C3_S10_L005,LAMI_RD308_D0_C3_S14_L005,LAMI_RD306_D23_C1_S11_L005,LAMI_RD308_D23_C2_S17_L005,LAMI_RD310_D7_C3_S24_L005,...,LAMI_RD304_D28_C3_S5_L005,LAMI_RD308_D0_C2_S13_L005,LAMI_RD310_D0_C2_S19_L005,LAMI_RD306_D11_C1_S8_L005,LAMI_RD310_D0_C3_S20_L005,LAMI_RD310_D14_C3_S21_L005,LAMI_RD306_D14_C1_S9_L005,LAMI_RD310_D21_C2_S23_L005,LAMI_RD308_D14_C3_S16_L005,LAMI_RD304_D0_C1_S1_L005
s__Cutibacterium acnes Type IA (MAG 1),777667,1513946,19467,127037,5905,1529048,63022,1186104,450121,19208,...,362236,1811,196880,825942,372845,441798,83571,196812,782381,111499
s__Cutibacterium acnes Type IA (MAG 2),775594,1522344,19719,127811,5994,1533464,63354,1197434,448513,19132,...,362942,1809,198453,832734,374919,447003,84130,197854,783524,111435
s__Cutibacterium acnes Type IA (MAG 3),788086,1544556,20344,131326,6190,1559337,63969,1210542,457334,19473,...,374395,1850,201364,844942,378610,457067,85865,201885,793252,114153
s__Cutibacterium acnes Type IA (MAG 4),780834,1535993,19895,128011,5951,1551302,63556,1199449,452006,19203,...,363097,1869,200068,839140,377336,451537,85088,197051,787833,111607
s__Cutibacterium acnes Type IA (MAG 5),901353,981993,15592,120927,6642,1020013,71516,1343335,516896,14697,...,421960,2453,152467,557413,273568,301208,55632,162549,870057,120580
s__Cutibacterium acnes Type IA (MAG 6),772539,1517019,19530,125088,5830,1528614,63224,1190296,446426,18995,...,356613,1782,195437,826306,368824,446245,83269,193424,777440,109765
s__Cutibacterium acnes Type IB (MAG 7),1198336,766189,16505,162529,9459,793535,84160,1280577,637993,14708,...,627653,3384,175952,436994,277259,262593,43103,199691,1099965,177293
s__Cutibacterium acnes Type IB (MAG 8),1270683,771643,17879,184999,10742,803676,88604,1310009,675108,16482,...,697855,3676,195881,443438,303719,278770,43281,226908,1166542,197660
s__Cutibacterium acnes Type IB (MAG 9),1215356,722464,15791,160657,9393,752914,84276,1248990,643518,14109,...,625667,3313,167269,416196,264908,250947,40790,191164,1112826,176692
s__Cutibacterium acnes Type IB (MAG 10),1224186,763109,16325,165837,9659,791185,85623,1281362,651978,14755,...,638399,3477,178087,436281,279675,261011,42871,201668,1120725,181934


In [289]:
# Calculate relative abundance by dividing each value by column sum
df = df.div(df.sum(axis=0))
df = df.transpose()

# Replace underscores with dots
df.index = df.index.str.replace('_', '.')

# remove everything after and including second to last period
df.index = df.index.to_series().str.split('.').str[:4].str.join('.')

# Map labels from metadata to df index using SampleID
df.index = df.index.map(md['label'])

df

,s__Cutibacterium acnes Type IA (MAG 1),s__Cutibacterium acnes Type IA (MAG 2),s__Cutibacterium acnes Type IA (MAG 3),s__Cutibacterium acnes Type IA (MAG 4),s__Cutibacterium acnes Type IA (MAG 5),s__Cutibacterium acnes Type IA (MAG 6),s__Cutibacterium acnes Type IB (MAG 7),s__Cutibacterium acnes Type IB (MAG 8),s__Cutibacterium acnes Type IB (MAG 9),s__Cutibacterium acnes Type IB (MAG 10),...,s__Cutibacterium acnes Type II (MAG 13),s__Cutibacterium acnes Type II (MAG 14),s__Cutibacterium acnes Type II (MAG 15),s__Cutibacterium acnes Type II (MAG 16),s__Cutibacterium acnes Type II (MAG 17),s__Cutibacterium acnes Type II (MAG 18),s__Cutibacterium acnes Type II (MAG 19),s__Cutibacterium acnes Type II (MAG 20),s__Cutibacterium acnes Type II (MAG 21),s__Cutibacterium acnes Type II (MAG 22)
PP_308_sev-2_NL,0.052598,0.052458,0.053302,0.052812,0.060963,0.052251,0.081050,0.085943,0.082201,0.082798,...,0.021692,0.028747,0.022334,0.023871,0.028958,0.021991,0.022148,0.022005,0.026350,0.022206
PP_306_sev-1_NL,0.111624,0.112244,0.113881,0.113250,0.072403,0.111851,0.056492,0.056894,0.053268,0.056265,...,0.006489,0.009440,0.006721,0.008647,0.010661,0.006864,0.006933,0.007231,0.012614,0.006727
PP_310_sev-5_L,0.032317,0.032736,0.033773,0.033028,0.025884,0.032422,0.027400,0.029681,0.026215,0.027101,...,0.056855,0.060435,0.057637,0.058077,0.068544,0.057141,0.059216,0.056674,0.081455,0.058539
PP_304_sev-0_NL,0.019606,0.019725,0.020268,0.019756,0.018663,0.019305,0.025083,0.028551,0.024794,0.025594,...,0.067717,0.072164,0.068848,0.069157,0.071016,0.068357,0.067217,0.064213,0.070943,0.067623
PP_304_sev-2_L,0.018022,0.018294,0.018892,0.018163,0.020272,0.017793,0.028869,0.032785,0.028668,0.029480,...,0.066980,0.072000,0.067477,0.068048,0.069952,0.067261,0.065060,0.062774,0.069998,0.065893
PP_306_sev-0_NL,0.110503,0.110822,0.112692,0.112111,0.073715,0.110472,0.057348,0.058081,0.054412,0.057178,...,0.006498,0.009600,0.006728,0.008463,0.010811,0.006717,0.006860,0.007088,0.012683,0.006599
PP_308_sev-0_NL,0.062576,0.062906,0.063517,0.063106,0.071010,0.062777,0.083565,0.087977,0.083680,0.085017,...,0.015382,0.022131,0.015461,0.017495,0.022653,0.015539,0.015795,0.015815,0.020517,0.015367
PP_306_sev-2_L,0.076128,0.076855,0.077696,0.076984,0.086219,0.076397,0.082191,0.084080,0.080164,0.082242,...,0.008993,0.014966,0.009406,0.011238,0.015600,0.009355,0.009516,0.009748,0.014870,0.009211
PP_308_sev-4_L,0.060658,0.060441,0.061630,0.060912,0.069657,0.060160,0.085976,0.090977,0.086720,0.087860,...,0.015077,0.021968,0.015653,0.017268,0.022359,0.015312,0.015554,0.015674,0.019977,0.015491
PP_310_sev-2_NL,0.029246,0.029130,0.029649,0.029238,0.022377,0.028921,0.022394,0.025095,0.021482,0.022466,...,0.061363,0.065380,0.062293,0.063635,0.069548,0.063147,0.064425,0.061104,0.079447,0.064725


In [290]:
# Create a mapping dictionary for PP numbers
unique_pp = sorted(list(set([idx.split('_')[0] + '_' + idx.split('_')[1] for idx in df.index])))
pp_map = {pp: f'PP_{i+1}' for i, pp in enumerate(unique_pp)}

# Apply the mapping to create new index
df.index = df.index.map(lambda x: pp_map[x.split('_')[0] + '_' + x.split('_')[1]] + '_' + '_'.join(x.split('_')[2:]))

# Sort index by PP number
df = df.sort_index()


df

,s__Cutibacterium acnes Type IA (MAG 1),s__Cutibacterium acnes Type IA (MAG 2),s__Cutibacterium acnes Type IA (MAG 3),s__Cutibacterium acnes Type IA (MAG 4),s__Cutibacterium acnes Type IA (MAG 5),s__Cutibacterium acnes Type IA (MAG 6),s__Cutibacterium acnes Type IB (MAG 7),s__Cutibacterium acnes Type IB (MAG 8),s__Cutibacterium acnes Type IB (MAG 9),s__Cutibacterium acnes Type IB (MAG 10),...,s__Cutibacterium acnes Type II (MAG 13),s__Cutibacterium acnes Type II (MAG 14),s__Cutibacterium acnes Type II (MAG 15),s__Cutibacterium acnes Type II (MAG 16),s__Cutibacterium acnes Type II (MAG 17),s__Cutibacterium acnes Type II (MAG 18),s__Cutibacterium acnes Type II (MAG 19),s__Cutibacterium acnes Type II (MAG 20),s__Cutibacterium acnes Type II (MAG 21),s__Cutibacterium acnes Type II (MAG 22)
PP_1_sev-0_NL,0.019606,0.019725,0.020268,0.019756,0.018663,0.019305,0.025083,0.028551,0.024794,0.025594,...,0.067717,0.072164,0.068848,0.069157,0.071016,0.068357,0.067217,0.064213,0.070943,0.067623
PP_1_sev-0_NL,0.014951,0.014888,0.015381,0.014915,0.017256,0.014620,0.026446,0.029734,0.026300,0.026935,...,0.069557,0.073563,0.070314,0.070921,0.072935,0.070105,0.068924,0.066152,0.071866,0.069583
PP_1_sev-1_NL,0.017463,0.017497,0.018049,0.017505,0.020343,0.017192,0.030259,0.033643,0.030163,0.030777,...,0.066149,0.070553,0.067103,0.067680,0.070027,0.066662,0.065689,0.062961,0.068973,0.066070
PP_1_sev-2_L,0.018022,0.018294,0.018892,0.018163,0.020272,0.017793,0.028869,0.032785,0.028668,0.029480,...,0.066980,0.072000,0.067477,0.068048,0.069952,0.067261,0.065060,0.062774,0.069998,0.065893
PP_1_sev-3_L,0.017640,0.017629,0.018059,0.017657,0.019076,0.017365,0.028048,0.031270,0.027953,0.028783,...,0.067108,0.070952,0.067638,0.068033,0.072810,0.067475,0.066604,0.063980,0.070932,0.067110
PP_1_sev-4_L,0.016416,0.016393,0.016751,0.016385,0.017564,0.016079,0.025857,0.028959,0.025526,0.026402,...,0.069038,0.072686,0.069714,0.070235,0.073542,0.069459,0.068344,0.065619,0.072391,0.069066
PP_2_sev-0_NL,0.110503,0.110822,0.112692,0.112111,0.073715,0.110472,0.057348,0.058081,0.054412,0.057178,...,0.006498,0.009600,0.006728,0.008463,0.010811,0.006717,0.006860,0.007088,0.012683,0.006599
PP_2_sev-1_L,0.111490,0.112236,0.114551,0.113514,0.074217,0.111087,0.057503,0.057740,0.054417,0.057193,...,0.006017,0.008784,0.006225,0.007922,0.010238,0.006322,0.006429,0.006650,0.011662,0.006125
PP_2_sev-1_NL,0.111624,0.112244,0.113881,0.113250,0.072403,0.111851,0.056492,0.056894,0.053268,0.056265,...,0.006489,0.009440,0.006721,0.008647,0.010661,0.006864,0.006933,0.007231,0.012614,0.006727
PP_2_sev-1_NL,0.104491,0.104935,0.106254,0.105701,0.068397,0.104709,0.054598,0.055044,0.051688,0.054277,...,0.011189,0.013920,0.011211,0.013107,0.015414,0.011286,0.011518,0.011622,0.017064,0.011265


In [292]:
# Define distinct colors manually
distinct_reds = ['#8B0000', '#B22222', '#DC143C', '#FF4500', '#CD5C5C', '#E9967A']
distinct_blues = ['#00008B', '#4169E1', '#1E90FF', '#00BFFF', '#87CEFA']
distinct_greens = ['#006400', '#228B22', '#32CD32', '#3CB371', '#66CDAA', '#8FBC8F', '#98FB98', '#00FA9A', '#00FF7F', '#7FFF00', '#ADFF2F']


# Assign strains
type_ia_cols = [col for col in df.columns if 'Type IA' in col]
type_ib_cols = [col for col in df.columns if 'Type IB' in col]
type_ii_cols = [col for col in df.columns if 'Type II' in col]

# Build color map
color_map = {}
for col, color in zip(type_ia_cols, distinct_reds[:len(type_ia_cols)]):
    color_map[col] = color
for col, color in zip(type_ib_cols, distinct_blues[:len(type_ib_cols)]):
    color_map[col] = color
for col, color in zip(type_ii_cols, distinct_greens[:len(type_ii_cols)]):
    color_map[col] = color

# Plot
fig, ax = plt.subplots(figsize=(10, 10))
df.plot(
    kind='bar',
    stacked=True,
    width=0.6,
    edgecolor='none',
    color=[color_map[col] for col in df.columns],
    ax=ax
)

# Customize
ax.set_title('Relative Abundance of C. acnes MAGs Across Samples', fontsize=18)
ax.set_xlabel('Sample by {ParticipantID}_{LocalLesionSeverity}_{Zone}', fontsize=14, y=-0.3)
ax.set_ylabel('Relative Abundance', fontsize=14)
ax.tick_params(axis='x', rotation=45, labelsize=12)
ax.tick_params(axis='y', labelsize=12)

# Create grouped legend entries
legend_elements = []

# Group label patches (invisible, just for spacing/headers)
legend_elements.append(Patch(facecolor='white', edgecolor='none', label='Type IA'))
legend_elements += [Patch(facecolor=color_map[col], label=col.replace("Cutibacterium acnes", "C. acnes")) for col in type_ia_cols]

legend_elements.append(Patch(facecolor='white', edgecolor='none', label='Type IB'))
legend_elements += [Patch(facecolor=color_map[col], label=col.replace("Cutibacterium acnes", "C. acnes")) for col in type_ib_cols]

legend_elements.append(Patch(facecolor='white', edgecolor='none', label='Type II'))
legend_elements += [Patch(facecolor=color_map[col], label=col.replace("Cutibacterium acnes", "C. acnes")) for col in type_ii_cols]


# Add legend below the plot
ax.legend(
    handles=legend_elements,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.3),
    ncol=3,
    fontsize=10,
    title_fontsize=7,
    frameon=True
)

plt.tight_layout()

# Save figure
plt.savefig('../Figures/metaG_Figures/metaG_relative-abundance_cacnes-strains22.png', dpi=600, bbox_inches='tight')
plt.savefig('../Figures/metaG_Figures/metaG_relative-abundance_cacnes-strains22.svg', dpi=600, bbox_inches='tight')
